In [1]:
%env CUDA_VISIBLE_DEVICES=2

env: CUDA_VISIBLE_DEVICES=2


### Wan-AI/Wan2.2-TI2V-5B

In [1]:
print("/data/gaoya/dataset/Caoza-PhysX-3D/PhysXNet/version_1/finaljson/48610.json")

/data/gaoya/dataset/Caoza-PhysX-3D/PhysXNet/version_1/finaljson/48610.json


In [ ]:
import os
import sys
import torch

sys.path.append("/home/gaoya/Code_Video/DiffSynth-Studio-main")


import os
import torch
from diffsynth import ModelConfig
from diffsynth.pipelines.wan_video import WanVideoPipeline

os.environ["DIFFSYNTH_SKIP_DOWNLOAD"] = "True"   # 本地模型时建议打开，避免还去远端补文件

base = "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B"

pipe = WanVideoPipeline.from_pretrained(
    torch_dtype=torch.bfloat16,
    device="cuda",
    model_configs=[
        # DiT：分片文件必须传 list
        ModelConfig(path=[
            f"{base}/diffusion_pytorch_model-00001-of-00003.safetensors",
            f"{base}/diffusion_pytorch_model-00002-of-00003.safetensors",
            f"{base}/diffusion_pytorch_model-00003-of-00003.safetensors",
        ]),
        # T5
        ModelConfig(path=f"{base}/models_t5_umt5-xxl-enc-bf16.pth"),
        # VAE
        ModelConfig(path=f"{base}/Wan2.2_VAE.pth"),
    ],
    # tokenizer 这里传目录
    tokenizer_config=ModelConfig(path=f"{base}/google/umt5-xxl"),
)

/data/gaoya/miniconda3/envs/wan/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
I0000 00:00:1774512662.433981  139523 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774512662.494263  139523 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774512663.824603  139523 port.cc:153] oneDNN custom operations are on. You may see slightly different n

Loading models from: [
    "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00001-of-00003.safetensors",
    "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00002-of-00003.safetensors",
    "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00003-of-00003.safetensors"
]
Loaded model: {
    "model_name": "wan_video_dit",
    "model_class": "diffsynth.models.wan_video_dit.WanModel",
    "extra_kwargs": {
        "has_image_input": false,
        "patch_size": [
            1,
            2,
            2
        ],
        "in_dim": 48,
        "dim": 3072,
        "ffn_dim": 14336,
        "freq_dim": 256,
        "text_dim": 4096,
        "out_dim": 48,
        "num_heads": 24,
        "num_layers": 30,
        "eps": 1e-06,
        "seperated_timestep": true,
        "require_clip_embedding": false,
        "require_vae_embedding": false,
        "fuse_vae_embedding_in_latents": true
    }
}
Loading models from: "/data/gaoya/ckpt/Wan-AI-Wan2

In [18]:
video_a = "/home/gaoya/Code_Video/Code_data/vis/output.mp4"
video_b = "/data/gaoya/AAA_test_video/Benchmark/physics_IQ/generated_videos/wan22_ti2v_5b/0001_perspective-left_trimmed-ball-and-block-fall.mp4"
import imageio.v2 as imageio
import numpy as np
resize_h = 720
resize_w = 1280
def preprocess_frame(frame_np: np.ndarray, target_h: int, target_w: int) -> torch.Tensor:
    img = Image.fromarray(frame_np).convert("RGB")
    img = img.resize((target_w, target_h), Image.BICUBIC)
    arr = np.asarray(img).astype(np.float32) / 255.0  # [H,W,3], 0~1
    ten = torch.from_numpy(arr).permute(2, 0, 1).contiguous()  # [3,H,W]
    return ten

def load_video_frames(path: str):
    reader = imageio.get_reader(path)
    frames = []
    for frame in reader:
        # frame: HWC uint8 RGB
        if frame.shape[-1] == 4:
            frame = frame[..., :3]
        frames.append(frame)
    reader.close()
    if len(frames) == 0:
        raise ValueError(f"视频为空: {path}")
    return frames
frames_a = load_video_frames(video_a)
frames_b = load_video_frames(video_b)
from einops import repeat, reduce

def preprocess_image(image, torch_dtype=None, device=None, pattern="B C H W", min_value=-1, max_value=1):
    # Transform a PIL.Image to torch.Tensor
    image = torch.Tensor(np.array(image, dtype=np.float32))
    image = image.to(dtype=torch_dtype, device=device)
    image = image * ((max_value - min_value) / 255) + min_value
    image = repeat(image, f"H W C -> {pattern}", **({"B": 1} if "B" in pattern else {}))
    return image
def preprocess_video(video, torch_dtype=None, device=None, pattern="B C T H W", min_value=-1, max_value=1):
    # Transform a list of PIL.Image to torch.Tensor
    video = [preprocess_frame(image, torch_dtype=torch_dtype, device=device, min_value=min_value, max_value=max_value) for image in video]
    video = torch.stack(video, dim=pattern.index("T") // 2)
    return video

ta = preprocess_video(frames_a, torch_dtype="bfloat16", device=pipe.device, pattern="B C T H W")
tb = preprocess_video(frames_b, torch_dtype="bfloat16", device=pipe.device, pattern="B C T H W")


input_latents = pipe.vae.encode_framewise(ta, device=pipe.device)




huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


TypeError: preprocess_frame() got an unexpected keyword argument 'torch_dtype'

In [19]:
import imageio.v2 as imageio
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

# =========================
# 路径
# =========================
video_a = "/home/gaoya/Code_Video/Code_data/vis/output.mp4"
video_b = "/data/gaoya/AAA_test_video/Benchmark/physics_IQ/generated_videos/wan22_ti2v_5b/0001_perspective-left_trimmed-ball-and-block-fall.mp4"

# =========================
# 分辨率
# 注意：最好是偶数，且建议能被 16 整除
# 720x1280 可以用，但显存占用较高
# =========================
resize_h = 720
resize_w = 1280

# =========================
# 工具函数
# =========================
def load_video_frames(path: str):
    reader = imageio.get_reader(path)
    frames = []
    for frame in reader:
        # frame: HWC uint8 RGB/RGBA
        if frame.shape[-1] == 4:
            frame = frame[..., :3]
        frames.append(frame)
    reader.close()
    if len(frames) == 0:
        raise ValueError(f"视频为空: {path}")
    return frames


def preprocess_frame(frame_np: np.ndarray, target_h: int, target_w: int) -> torch.Tensor:
    """
    输入:
        frame_np: [H, W, 3], uint8, RGB
    输出:
        tensor: [3, H, W], float32, range [-1, 1]
    """
    img = Image.fromarray(frame_np).convert("RGB")
    img = img.resize((target_w, target_h), Image.BICUBIC)
    arr = np.asarray(img).astype(np.float32) / 255.0   # [H, W, 3], 0~1
    arr = arr * 2.0 - 1.0                              # [-1, 1]
    ten = torch.from_numpy(arr).permute(2, 0, 1).contiguous()  # [3, H, W]
    return ten


def build_video_tensor(frames, target_h: int, target_w: int) -> torch.Tensor:
    """
    输入:
        frames: list of np.ndarray
    输出:
        video_tensor: [1, 3, T, H, W]
    """
    tchw = torch.stack([preprocess_frame(f, target_h, target_w) for f in frames], dim=0)  # [T, 3, H, W]
    video_5d = tchw.permute(1, 0, 2, 3).unsqueeze(0).contiguous()                          # [1, 3, T, H, W]
    return video_5d


@torch.no_grad()
def encode_video_framewise(video_5d: torch.Tensor, pipe):
    """
    输入:
        video_5d: [1, 3, T, H, W], float32 in [-1, 1]
    输出:
        latents
    """
    return pipe.vae.encode_framewise(video_5d, device=pipe.device)


def latents_to_frame_features(latents: torch.Tensor) -> torch.Tensor:
    """
    把 VAE latent 转成逐帧特征 [T, D]
    兼容常见形状:
      [B, C, T, H, W]
      [B, C, T]
    """
    if latents.ndim == 5:
        # [B, C, T, H, W] -> [T, C, H, W] -> [T, D]
        x = latents[0].permute(1, 0, 2, 3).contiguous()
        x = x.flatten(1)
        return x
    elif latents.ndim == 3:
        # [B, C, T] -> [T, C]
        x = latents[0].transpose(0, 1).contiguous()
        return x
    else:
        raise ValueError(f"不支持的 latent shape: {tuple(latents.shape)}")


def cosine_similarity_diag(feat_a: torch.Tensor, feat_b: torch.Tensor) -> torch.Tensor:
    """
    同索引帧 cosine
    feat_a, feat_b: [T, D]
    """
    feat_a = F.normalize(feat_a, dim=1)
    feat_b = F.normalize(feat_b, dim=1)
    diag_cos = (feat_a * feat_b).sum(dim=1)   # [T]
    return diag_cos


def cosine_similarity_matrix(feat_a: torch.Tensor, feat_b: torch.Tensor) -> torch.Tensor:
    """
    全帧两两 cosine 相似度矩阵
    feat_a, feat_b: [T, D]
    return: [T, T]
    """
    feat_a = F.normalize(feat_a, dim=1)
    feat_b = F.normalize(feat_b, dim=1)
    sim = feat_a @ feat_b.T
    return sim


def topk_pairs_from_sim(sim: torch.Tensor, k: int = 20):
    """
    从相似度矩阵里找 top-k 最相似帧对
    """
    T_a, T_b = sim.shape
    flat = sim.flatten()
    k = min(k, flat.numel())
    vals, idx = torch.topk(flat, k=k)

    pairs = []
    for v, ind in zip(vals.tolist(), idx.tolist()):
        i = ind // T_b
        j = ind % T_b
        pairs.append((i, j, v))
    return pairs


def save_similarity_heatmap(sim: torch.Tensor, save_path: str):
    plt.figure(figsize=(8, 6))
    plt.imshow(sim.cpu().numpy(), aspect="auto")
    plt.colorbar()
    plt.xlabel("video_b frame index")
    plt.ylabel("video_a frame index")
    plt.title("VAE Feature Cosine Similarity Matrix")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


# =========================
# 1. 读取视频
# =========================
frames_a = load_video_frames(video_a)
frames_b = load_video_frames(video_b)

print(f"len(frames_a) = {len(frames_a)}")
print(f"len(frames_b) = {len(frames_b)}")

# =========================
# 2. 截成相同帧数
# =========================
min_len = min(len(frames_a), len(frames_b))
frames_a_trim = frames_a[:min_len]
frames_b_trim = frames_b[:min_len]

print(f"trimmed length = {min_len}")

# =========================
# 3. 构造 [1, 3, T, H, W]
# =========================
ta = build_video_tensor(frames_a_trim, resize_h, resize_w)
tb = build_video_tensor(frames_b_trim, resize_h, resize_w)

print("ta.shape =", ta.shape)  # [1, 3, T, H, W]
print("tb.shape =", tb.shape)

# =========================
# 4. VAE 编码
# 这里默认你已经提前有 pipe
# 例如:
# pipe = WanVideoPipeline.from_pretrained(...)
# =========================
with torch.no_grad():
    latents_a = encode_video_framewise(ta.to(dtype = torch.bfloat16), pipe)
    latents_b = encode_video_framewise(tb.to(dtype = torch.bfloat16), pipe)

print("latents_a.shape =", latents_a.shape)
print("latents_b.shape =", latents_b.shape)

# =========================
# 5. 转成逐帧特征 [T, D]
# =========================
feat_a = latents_to_frame_features(latents_a)
feat_b = latents_to_frame_features(latents_b)

print("feat_a.shape =", feat_a.shape)
print("feat_b.shape =", feat_b.shape)

# =========================
# 6. 同索引帧相似度
# =========================
diag_cos = cosine_similarity_diag(feat_a, feat_b)

print("\n===== 同索引帧 cosine =====")
print("diag_cos.shape =", diag_cos.shape)
print("diag mean =", diag_cos.mean().item())
print("diag min  =", diag_cos.min().item())
print("diag max  =", diag_cos.max().item())
print("diag std  =", diag_cos.std().item())

# =========================
# 7. 全帧两两相似度矩阵
# =========================
sim = cosine_similarity_matrix(feat_a, feat_b)
print("\n===== 全帧相似度矩阵 =====")
print("sim.shape =", sim.shape)

# 每个 A 帧在 B 里最像哪一帧
best_vals, best_idx = sim.max(dim=1)

print("\n===== A 中每一帧在 B 中最相似匹配 =====")
for i in range(min(20, len(best_idx))):
    print(f"A[{i}] -> B[{best_idx[i].item()}], cos={best_vals[i].item():.6f}")

# top-k 最相似帧对
top_pairs = topk_pairs_from_sim(sim, k=20)
print("\n===== Top-20 最相似帧对 =====")
for rank, (i, j, v) in enumerate(top_pairs, 1):
    print(f"{rank:02d}. A[{i}] <-> B[{j}], cos={v:.6f}")

# =========================
# 8. 保存热力图
# =========================
heatmap_path = "./vae_similarity_heatmap.png"
save_similarity_heatmap(sim, heatmap_path)
print(f"\n相似度热力图已保存到: {heatmap_path}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


len(frames_a) = 121
len(frames_b) = 120
trimmed length = 120
ta.shape = torch.Size([1, 3, 120, 720, 1280])
tb.shape = torch.Size([1, 3, 120, 720, 1280])
latents_a.shape = torch.Size([1, 48, 120, 45, 80])
latents_b.shape = torch.Size([1, 48, 120, 45, 80])
feat_a.shape = torch.Size([120, 172800])
feat_b.shape = torch.Size([120, 172800])

===== 同索引帧 cosine =====
diag_cos.shape = torch.Size([120])
diag mean = 0.8203125
diag min  = 0.796875
diag max  = 0.83984375
diag std  = 0.01068115234375

===== 全帧相似度矩阵 =====
sim.shape = torch.Size([120, 120])

===== A 中每一帧在 B 中最相似匹配 =====
A[0] -> B[0], cos=0.832031
A[1] -> B[1], cos=0.828125
A[2] -> B[2], cos=0.824219
A[3] -> B[3], cos=0.820312
A[4] -> B[4], cos=0.812500
A[5] -> B[5], cos=0.824219
A[6] -> B[6], cos=0.820312
A[7] -> B[7], cos=0.808594
A[8] -> B[8], cos=0.804688
A[9] -> B[9], cos=0.808594
A[10] -> B[10], cos=0.808594
A[11] -> B[11], cos=0.804688
A[12] -> B[12], cos=0.804688
A[13] -> B[13], cos=0.804688
A[14] -> B[14], cos=0.808594
A[15] -

TypeError: Got unsupported ScalarType BFloat16

<Figure size 800x600 with 0 Axes>

In [3]:
from PIL import Image
from diffsynth.utils.data import save_video
input_image = Image.open(
    "/data/gaoya/dataset/physics-iq-benchmark/switch-frames/0001_switch-frames_anyFPS_perspective-left_trimmed-ball-and-block-fall.jpg"
    ).resize((1280, 720))
video = pipe(
    prompt="Two pillows on a table and two grabber tools hanging above them from which a brown tennis ball and an orange block are suspended. The grabber tools let go of the ball and block. Static shot with no camera movement.",
    negative_prompt="",
    seed=42, 
    
    height=720, width=1280,
    input_image=input_image,
    num_frames=240,
    cfg_scale=5.0,
    num_inference_steps=50,
)
save_video(video, "/home/gaoya/Code_Video/Code_data/vis/output.mp4", fps=30)

height % 32 != 0. We round it up to 736.


Saving video:   0%|          | 0/121 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Saving video: 100%|██████████| 121/121 [00:00<00:00, 169.17it/s]
